In [2]:
import numpy as np 
from sentence_transformers import SentenceTransformer

embed_model=SentenceTransformer("all-MiniLM-L6-v2")

def cosine_similarity(a,b):
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))


words=["automobile", "vehicle", "dinosaur", "stick"]
embeddings={word: embed_model.encode(word) for word in words}

print("Similarity (automobile vs vehicle):",
      cosine_similarity(embeddings["automobile"], embeddings["vehicle"]))
print("Similarity (automobile vs dinosaur):", 
      cosine_similarity(embeddings["automobile"],embeddings["dinosaur"]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Similarity (automobile vs vehicle): 0.84713644
Similarity (automobile vs dinosaur): 0.39764318


In [3]:
import json 
import pandas as pd

with open("embedding_index_3m.json", "r") as f:
    data=json.load(f)
    df=pd.DataFrame(data)

    print("generating local embedding for transcripts....")
    embed_model=SentenceTransformer("all-MiniLM-L6-v2")
    df["local_vector"]=list(
        embed_model.encode(df["summary"].tolist(), show_progress_bar=True)

    )

    def cosine_similarity(a, b):
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

    def get_videos(query,top_k=3):
        query_vector=embed_model.encode(query)

        df["similarity"]=df["local_vector"].apply(
            lambda x: cosine_similarity(query_vector,x)
        )

        results=df.sort_values("similarity",ascending=False).head(top_k)
        return results 

generating local embedding for transcripts....


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/45 [00:00<?, ?it/s]

In [ ]:
import os
from groq import Groq

groq_client=Groq(api_key=os.environ.get("GROQ_API_KEY"))

def search_and_summarize(query):
    results=get_videos(query,top_k=2)

    context=""

    print("\n--- MATCHING YOUTUBE CLIPS FOUND ---")

    for idx, row in results.iterrows():
        url = f"https://www.youtube.com/watch?v={row['videoId']}&t={row['start']}s"
        print(f"Title: {row['title']}")
        print(f"Link: {url}")
        print(f"Similarity Score: {row['similarity']:.4f}\n")
        context += f"Video Title: {row['title']}\nSummary: {row['summary']}\nURL: {url}\n\n"

    prompt = f"""
    You are an AI assistant answering questions based on YouTube video transcripts.
    
    User Query: "{query}"
    
    Retrieved Context:
    {context}
    
    Please provide a concise answer to the user's question and cite the relevant video links.
    """

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        max_tokens=512
    )
    return response.choices[0].message.content
